<a href="https://colab.research.google.com/github/SampurnaKumar-2007/Automated-sales-analytics/blob/main/automated_sales_analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Installs external openpyxl package so Python can generate and save Excel files
!pip install openpyxl

# Imports pandas
import pandas as pd
import numpy as np

print("Libraries are imported succesfully!!")

Libraries are imported succesfully!!


In [4]:
# Sets a random seed so the numbers are predictable
np.random.seed(0) #accepts any integer from 0 to 2^32-1

# Generate raw data (500 orders)
n_rows = 500
dates = pd.date_range(start="2025-01-01", periods=180, freq="D") #freq=D tells panda to step forwards day-by-day

raw_data = {
    "Order_ID": [f"Ord-{1000 + i}" for i in range(n_rows)],
    "Date": np.random.choice(dates, size=n_rows),
    "Region": np.random.choice(["North", "South", "East", "West"], size=n_rows, p=[0.3, 0.25, 0.25, 0.2]),#p here is the probability of region chosen
    "Product_Category": np.random.choice(["Electronics", "Clothing", "Home & Kitchen", "Books"], size=n_rows),
    "Units_Sold": np.random.randint(1, 20, size=n_rows),#this is wriiten in format (low,high) so if high unit is 20 , 20 is not included in the random generation
    "Unit_Price": np.random.choice([15.0, 55.5, 49.99, 12.0, 1500.90, 2000.00,2230.99], size=n_rows),
    "Customer_Rating": np.random.choice([1.0, 2.0, 3.0, 4.0, 5.0, np.nan], size=n_rows, p=[0.05, 0.1, 0.2, 0.35, 0.25, 0.05]) #p is the probablity of respective ratings
}

# Converts dictionary to Pandas DataFrame
df_raw = pd.DataFrame(raw_data)

# Inject some missing values into 'Units_Sold' to simulate real messy data
df_raw.loc[df_raw.sample(35).index, "Units_Sold"] = np.nan

# Save as a raw CSV file
df_raw.to_csv("raw_sales_data.csv", index=False) #index=false tells panda not to write default row index numbers as seperate column in generated CSV files

print("Raw dataset has been created successfully and saved as 'raw_sales_data.csv'!!")

Raw dataset has been created successfully and saved as 'raw_sales_data.csv'!!


In [5]:
# 1. Loading the CSV file into a Pandas DataFrame
df = pd.read_csv("raw_sales_data.csv")

# 2. Check initial missing values
print("--- Missing Values Before Cleaning ---")
print(df.isnull().sum())
print("\n" + "=" * 40 + "\n")

# 3. Clean Missing Data (Imputation)
# Fill missing Units_Sold with median
df["Units_Sold"] = df["Units_Sold"].fillna(df["Units_Sold"].median())

# Fill missing Customer_Rating with average (rounded to 1 decimal place)
df["Customer_Rating"] = df["Customer_Rating"].fillna(
    round(df["Customer_Rating"].mean(), 1)
)

# 4. Feature Engineering: Compute Total Revenue for each order
df["Total_Revenue"] = df["Units_Sold"] * df["Unit_Price"]

# 5. Format Date column
df["Date"] = pd.to_datetime(df["Date"])
df["Month"] = df["Date"].dt.strftime("%Y-%m")

print("--- Missing Values After Cleaning ---")
print(df.isnull().sum())
print("\n" + "=" * 40 + "\n")

# Display the first 5 rows of our cleaned data
print("--- Cleaned Sample Data ---")
df.head()

--- Missing Values Before Cleaning ---
Order_ID             0
Date                 0
Region               0
Product_Category     0
Units_Sold          35
Unit_Price           0
Customer_Rating     38
dtype: int64


--- Missing Values After Cleaning ---
Order_ID            0
Date                0
Region              0
Product_Category    0
Units_Sold          0
Unit_Price          0
Customer_Rating     0
Total_Revenue       0
Month               0
dtype: int64


--- Cleaned Sample Data ---


,Order_ID,Date,Region,Product_Category,Units_Sold,Unit_Price,Customer_Rating,Total_Revenue,Month
0,Ord-1000,2025-06-22,North,Home & Kitchen,14.0,55.50,3.7,777.00,2025-06
1,Ord-1001,2025-02-17,East,Clothing,13.0,1500.90,5.0,19511.70,2025-02
2,Ord-1002,2025-04-28,North,Home & Kitchen,6.0,2000.00,3.0,12000.00,2025-04
3,Ord-1003,2025-03-09,South,Books,9.0,49.99,3.7,449.91,2025-03
4,Ord-1004,2025-04-14,South,Electronics,9.0,15.00,5.0,135.00,2025-04


In [6]:
# 1. Category-wise Summary
category_summary = (
    df.groupby("Product_Category")
    .agg(
        Total_Orders=("Order_ID", "count"),
        Total_Units=("Units_Sold", "sum"),
        Total_Revenue=("Total_Revenue", "sum"),
        Avg_Rating=("Customer_Rating", "mean"),
    )
    .reset_index()
    .round(2)
)

# 2. Region-wise Summary
region_summary = (
    df.groupby("Region")
    .agg(
        Total_Orders=("Order_ID", "count"),
        Total_Revenue=("Total_Revenue", "sum"),
    )
    .reset_index()
    .sort_values(by="Total_Revenue", ascending=False)
    .round(2)
)

print("--- Category Performance Summary ---")
print(category_summary)
print("\n" + "=" * 40 + "\n")

print("--- Regional Sales Performance ---")
print(region_summary)

--- Category Performance Summary ---
  Product_Category  Total_Orders  Total_Units  Total_Revenue  Avg_Rating
0            Books           122       1215.0      942744.48        3.78
1         Clothing           124       1304.0     1188014.82        3.69
2      Electronics           137       1438.0     1221618.12        3.73
3   Home & Kitchen           117       1136.0      967661.24        3.72


--- Regional Sales Performance ---
  Region  Total_Orders  Total_Revenue
1  North           154     1496669.34
3   West           106     1006845.24
2  South           140      995599.33
0   East           100      820924.75


In [7]:
# Define output filename
output_filename = "Automated_Sales_Report.xlsx"

# Use ExcelWriter with openpyxl engine to write to multiple sheets
with pd.ExcelWriter(output_filename, engine="openpyxl") as writer:

    # Sheet 1: Executive Summary (Category & Regional Performance)
    category_summary.to_excel(
        writer, sheet_name="Executive Summary", startrow=1, index=False
    )
    region_summary.to_excel(
        writer, sheet_name="Executive Summary", startrow=10, index=False
    )

    # Sheet 2: Cleaned Transactional Data
    df.to_excel(writer, sheet_name="Cleaned Data", index=False)

print(
    f"Success! Report generated and saved as '{output_filename}' in your files."
)

Success! Report generated and saved as 'Automated_Sales_Report.xlsx' in your files.
